# 00.03 · Your first LLM call (free)

**Chapter:** Setup · **Milestone:** M0

Many providers expose an **OpenAI-compatible** endpoint, so one SDK (`openai`) talks to all of them by changing `base_url`. Chapter 05 covers native SDKs (`anthropic`, `google-genai`) and gateways (LiteLLM).

## Get keys (all free, no card needed at time of writing — limits change, check each site)
| Provider | Free? | Where | Env var |
|---|---|---|---|
| Google Gemini (Flash models) | free tier | https://aistudio.google.com/apikey | `GEMINI_API_KEY` |
| Groq (open models, very fast) | free tier | https://console.groq.com/keys | `GROQ_API_KEY` |
| OpenRouter (`:free` models) | free models, daily cap | https://openrouter.ai/keys | `OPENROUTER_API_KEY` |
| Ollama (local) | free, runs on your Mac | https://ollama.com — then `ollama pull qwen3:1.7b` | none |
| OpenAI / Anthropic | **paid** (pay-as-you-go, a few $ go far) | platform.openai.com · console.anthropic.com | `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` |

Put keys in `.env` (git-ignored). Never paste keys into notebooks.

> Free tiers may use your prompts for training. Don't send private/customer data to them.


In [4]:
import os, time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# name: (base_url, api-key env var, model). Model ids change often: check each provider's model list.
PROVIDERS = {
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GEMINI_API_KEY", "gemini-flash-lite-latest"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "openai/gpt-oss-120b"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "openrouter/free"),
    "ollama":     ("http://localhost:11434/v1", None, os.getenv("OLLAMA_MODEL", "qwen3:1.7b")),
}

def ask(provider, prompt):
    base_url, key_var, model = PROVIDERS[provider]
    client = OpenAI(base_url=base_url, api_key=os.getenv(key_var) if key_var else "ollama", timeout=60)
    start = time.time()
    kwargs = {}
    if client.base_url.host == "localhost":
        kwargs["reasoning_effort"] = "none"  # qwen3 thinks by default; skip that on a laptop
    resp = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], **kwargs)
    return resp.choices[0].message.content, resp.usage, time.time() - start


In [5]:
prompt = "Who are the main characters in Jujutsu Kaisen from the mangaka Gege Akutami?"

for name, (_, key_var, _) in PROVIDERS.items():
    if key_var and not os.getenv(key_var):
        print(f"--- {name}: skipped (no {key_var} in .env)")
        continue
    try:
        text, usage, secs = ask(name, prompt)
        print(f"--- {name} ({secs:.1f}s, {usage.prompt_tokens}+{usage.completion_tokens} tokens)\n{text}\n")
    except Exception as e:  # provider down, model renamed, Ollama not running...
        print(f"--- {name}: error: {e}\n")


--- gemini (5.3s, 21+909 tokens)
In *Jujutsu Kaisen*, created by Gege Akutami, the story revolves around a world where cursed energy flows from humans, and individuals known as Jujutsu Sorcerers fight against malevolent entities called Curses. 

Here are the main characters of the series, categorized by their roles:

### The Main Trio (Students)
*   **Yuji Itadori:** The protagonist of the series. Yuji is a kind-hearted high schooler with incredible physical abilities. His life changes forever when he swallows a rotting finger belonging to Ryomen Sukuna—the "King of Curses"—to save his friends. He becomes Sukuna's vessel and joins Tokyo Jujutsu High to find all of Sukuna's fingers and ensure a "proper death."
*   **Megumi Fushiguro:** A taciturn and pragmatic first-year student at Tokyo Jujutsu High. Megumi is a sorcerer from the prestigious Zen'in Clan who uses the "Ten Shadows Technique," allowing him to summon various shikigami using shadows. He is the one who originally tracked dow

## Exercises
1. Get at least **two** providers working (Milestone M0).
2. Ask all providers the same tricky question; compare quality, speed and token counts.
3. Add `stream=True` and print tokens as they arrive.
4. Add a `system` message that forces answers in your native language.


In [6]:
# Exercise 3 — stream tokens as they arrive
# Run the PROVIDERS cell first. Swap provider to gemini / openrouter / ollama if you want.

provider = "groq"
prompt = "Who are the main characters in Jujutsu Kaisen from the mangaka Gege Akutami?"

base_url, key_var, model = PROVIDERS[provider]
client = OpenAI(
    base_url=base_url,
    api_key=os.getenv(key_var) if key_var else "ollama",
    timeout=60,
)

print(f"--- {provider} (streaming)\n")
stream = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    stream=True,
)
for chunk in stream:
    piece = chunk.choices[0].delta.content  # one token (or a few), or None
    if piece:
        print(piece, end="", flush=True)
print()

--- groq (streaming)

**Jujutsu Kaisen** — created by mangaka **Gege Akutami** — revolves around a core cast of sorcerers, cursed spirits, and supporting figures. Below are the primary characters most central to the story (as of the main manga arcs up through the “Shibuya Incident” and “Culling Game” arcs).

| Character | Role / Title | Brief Description | Notable Abilities / Cursed Techniques |
|-----------|--------------|-------------------|----------------------------------------|
| **Yuji Itadori** | Protagonist / First‑Year Student at Tokyo Jujutsu High | A naturally athletic teen with a strong sense of justice. He becomes a Jujutsu sorcerer after swallowing **Sukuna’s** (the King of Curses) finger to save his friends. | **Superhuman physicality**, **Reverse Cursed Technique** (self‑healing), **Black Flash** (temporarily amplifies strike power). He can also channel Sukuna’s power, though he rarely does. |
| **Megumi Fushiguro** | First‑Year Student, Team‑Leader for the “Good Luck”

In [7]:
# Exercise 4 — system message forces the answer language
# Change `language` if yours is not Spanish. The user prompt stays in English on purpose.

provider = "groq"
language = "español"
prompt = "Who are the main characters in Jujutsu Kaisen from the mangaka Gege Akutami?"

base_url, key_var, model = PROVIDERS[provider]
client = OpenAI(
    base_url=base_url,
    api_key=os.getenv(key_var) if key_var else "ollama",
    timeout=60,
)

resp = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": f"Always answer in {language}. Never use English."},
        {"role": "user", "content": prompt},
    ],
)
print(resp.choices[0].message.content)

Los personajes principales de **Jujutsu Kaisen**, creados por el mangaka Gege Akutami, son:

| Personaje | Breve descripción |
|-----------|-------------------|
| **Yuji Itadori** | Protagonista de la serie. Un estudiante de secundaria de gran corazón y fuerza física extraordinaria que se convierte en hechicero después de ingerir un dedo de Sukuna. |
| **Satoru Gojo** | El hechicero más poderoso de la escuela Jujutsu. Posee la técnica de energía infinita y la habilidad de manipular el espacio con su Técnica de Dominio Infinito. Es mentor de los estudiantes de primer año. |
| **Megumi Fushiguro** | Compañero de clase de Itadori. Posee la Técnica de la Invocación de Shikigami, que le permite controlar criaturas espirituales para el combate. |
| **Nobara Kugisaki** | Hechicera de origen de la prefectura de Osaka. Utiliza muñecas malditas y clavos con energía maldita como sus principales armas. |
| **Ryomen Sukuna** | Conocido como el “Rey de las Maldiciones”. Su cuerpo es el receptáculo d

## Checkpoint (M0)
- [ ] Environment check passes
- [ ] Two providers answer from this notebook
- [ ] Repo is on GitHub